In [1]:
!git clone https://github.com/DayvisonGomes/desafio_dhauz.git

Cloning into 'desafio_dhauz'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 125 (delta 51), reused 104 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 312.28 KiB | 13.01 MiB/s, done.
Resolving deltas: 100% (51/51), done.


# Fluxo completo (Colab) — Treino, Vector DB e Demo

Este notebook guia a execução passo-a-passo no Colab: instalar dependências, baixar dados do Kaggle, treinar DistilBERT, construir o Chroma DB, e lançar a interface (com opção de LLM remoto).


In [1]:
# 1) Instalar dependências
%cd desafio_dhauz
!pip install -r requirements.txt

/content/desafio_dhauz


## 2) Upload das credenciais do Kaggle (se necessário)
No painel do Colab, faça upload do arquivo `kaggle.json` para `/root/.kaggle/kaggle.json` ou use o método que preferir.


In [2]:
# 3) Baixar e pré-processar dados
!python scripts/download_data.py


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
100% 3.45M/3.45M [00:00<00:00, 182MB/s]
Extracting files...
Saved to data/dataset_processed.csv


In [6]:
# 4) Treinar DistilBERT (padrão: 2 épocas; ajuste se necessário)
!python scripts/train_distilbert.py


Map: 100% 38269/38269 [00:10<00:00, 3514.36 examples/s]
Map: 100% 9568/9568 [00:02<00:00, 4081.91 examples/s]
Loading weights: 100% 100/100 [00:00<00:00, 904.22it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

In [4]:
# 5) Construir Chroma DB (embeddings)
!python scripts/build_vector_db.py


/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
modules.json: 100% 349/349 [00:00<00:00, 1.63MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 491kB/s]
README.md: 10.5kB [00:00, 25.0MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 259kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.91MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 64.3MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 1474.28it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-

## 6) (Opcional) Exportar Chroma DB para ZIP e transferir para outro host
Use `--export-chroma` para empacotar o `data/chroma_db` em um ZIP para download.


In [ ]:
# Export chroma db to zip (opcional)
!python scripts/demo.py --chroma-dir data/chroma_db --export-chroma data/chroma_db.zip


# Teste demo agent

In [12]:
!python scripts/demo_agent.py --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --mode hybrid --use-llm --llm-model Qwen/Qwen2.5-3B-Instruct


Loading DistilBERT...
Loading weights: 100% 104/104 [00:00<00:00, 755.68it/s, Materializing param=pre_classifier.weight]
DistilBERT loaded
Detected classes: ['Access', 'Administrative rights', 'HR Support', 'Hardware', 'Internal Project', 'Miscellaneous', 'Purchase', 'Storage']
/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100% 103/103 [00:00<00:00, 1186.35it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
-------

## 7) Rodar demo sem LLM (seguro)
Inicie a interface Gradio local sem carregar LLM pesado:


In [ ]:
!python scripts/demo.py --mode hybrid --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --use-llm --llm-model Qwen/Qwen2.5-3B-Instruct


/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100% 103/103 [00:00<00:00, 2171.16it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading existing Chroma DB from data/chroma_db
/content/desafio_dhauz/dhauz_tick

## 8) Rodar demo usando LLM remoto (HuggingFace Inference API)
Defina `REMOTE_LLM_URL` e `REMOTE_LLM_KEY` no ambiente ou passe como argumentos. Exemplo usando HuggingFace Inference API:


In [ ]:
# Exemplo: export HF inference URL e KEY (substitua pelos seus valores)
# export REMOTE_LLM_KEY="hf_xxx"
# REMOTE_LLM_URL should be like https://api-inference.huggingface.co/models/your-model
# Execute the demo pointing to the remote LLM:
!python scripts/demo.py --use-llm-remote --remote-llm-url "https://api-inference.huggingface.co/models/Qwen/Qwen2.5-3B-Instruct" --remote-llm-key $REMOTE_LLM_KEY --mode hybrid



# Avaliar DistilBERT (val por padrão, salva em ./results)


In [ ]:
!python scripts/evaluate.py --processed data/dataset_processed.csv --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --out-dir /content/desafio_dhauz/dhauz_ticket_classifier/results

Loading weights: 100% 104/104 [00:00<00:00, 893.96it/s, Materializing param=pre_classifier.weight]
Batches: 100% 7/7 [00:01<00:00,  5.10it/s]
                       precision    recall  f1-score   support

               Access     0.9118    0.9688    0.9394        32
Administrative rights     0.8333    0.7143    0.7692         7
           HR Support     0.9038    0.9038    0.9038        52
             Hardware     0.8491    0.8824    0.8654        51
     Internal Project     0.9231    1.0000    0.9600        12
        Miscellaneous     0.8696    0.7692    0.8163        26
             Purchase     1.0000    0.8750    0.9333         8
              Storage     0.9167    0.9167    0.9167        12

             accuracy                         0.8900       200
            macro avg     0.9009    0.8788    0.8880       200
         weighted avg     0.8900    0.8900    0.8890       200

Saved heatmap: /content/desafio_dhauz/dhauz_ticket_classifier/results/confusion_matrix_heatmap.png


In [ ]:
!python scripts/evaluate.py --processed data/dataset_processed.csv --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --out-dir /content/desafio_dhauz/dhauz_ticket_classifier/results --use-train

Loading weights: 100% 104/104 [00:00<00:00, 908.28it/s, Materializing param=pre_classifier.weight]
Batches: 100% 7/7 [00:01<00:00,  3.93it/s]
                       precision    recall  f1-score   support

               Access     1.0000    1.0000    1.0000        24
Administrative rights     1.0000    0.8571    0.9231         7
           HR Support     0.9565    0.9167    0.9362        48
             Hardware     0.9565    0.9565    0.9565        69
     Internal Project     0.7500    0.6000    0.6667         5
        Miscellaneous     0.9643    0.9643    0.9643        28
             Purchase     0.7778    1.0000    0.8750         7
              Storage     0.8571    1.0000    0.9231        12

             accuracy                         0.9450       200
            macro avg     0.9078    0.9118    0.9056       200
         weighted avg     0.9470    0.9450    0.9447       200

Saved heatmap: /content/desafio_dhauz/dhauz_ticket_classifier/results/confusion_matrix_heatmap.png


# Avaliar RAG/Hybrid (val por padrão). Modo: rag ou hybrid


In [ ]:
!python scripts/evaluate_rag.py --processed data/dataset_processed.csv --chroma-dir data/chroma_db --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --out-dir /content/desafio_dhauz/dhauz_ticket_classifier/results --mode hybrid --use-llm --llm-model Qwen/Qwen2.5-3B-Instruct


/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100% 103/103 [00:00<00:00, 953.70it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:26: LangChainD

In [ ]:
!python scripts/evaluate_rag.py --processed data/dataset_processed.csv --chroma-dir data/chroma_db --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --out-dir /content/desafio_dhauz/dhauz_ticket_classifier/results --mode hybrid --use-llm --llm-model Qwen/Qwen2.5-3B-Instruct --use-train


/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100% 103/103 [00:00<00:00, 882.29it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:26: LangChainD

In [ ]:
!python scripts/evaluate_rag.py --processed data/dataset_processed.csv --chroma-dir data/chroma_db --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --out-dir /content/desafio_dhauz/dhauz_ticket_classifier/results --mode rag --use-llm --llm-model Qwen/Qwen2.5-3B-Instruct


/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100% 103/103 [00:00<00:00, 1043.56it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:26: LangChain

In [ ]:
!python scripts/evaluate_rag.py --processed data/dataset_processed.csv --chroma-dir data/chroma_db --checkpoint /content/desafio_dhauz/dhauz_ticket_classifier/results --out-dir /content/desafio_dhauz/dhauz_ticket_classifier/results --mode rag --use-llm --llm-model Qwen/Qwen2.5-3B-Instruct --use-train


/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
Loading weights: 100% 103/103 [00:00<00:00, 2314.06it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/desafio_dhauz/dhauz_ticket_classifier/rag/vector_store.py:26: LangChain

# Avaliar RAG com LLM remoto (HuggingFace Inference API)


In [ ]:
!python scripts/evaluate_rag.py --use-llm-remote --remote-llm-url "https://api-inference.huggingface.co/models/SEU-MODELO" --remote-llm-key $HF_KEY

## Observações finais
- Se ocorrer OOM ao usar modelos locais, desative `--use-llm` e prefira `--use-llm-remote` com um endpoint gerenciado.
- Ajuste `--mode` entre `rag` e `hybrid` dependendo do comportamento desejado.

